# IT160IU — Data Mining (Fall 2026)
# Step 1 (Python part): Data Cleaning

**Role:** Person 1 — Dataset & Pre-processing (Step 1, 20 pts)

**Scope of this notebook** (per instructor's guidance): Python/pandas is used only for
**data cleaning** (handling duplicates and missing values) and to **export a clean CSV**.
**Data Analysis and ARFF export are done in Java** — see `DataAnalysis.java` in this same
folder, which reads the CSV this notebook produces.

**Deliverable:** `cleaned_heart_disease.csv`

In [1]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
# DATA_PATH accepts both a relative and an absolute path. Default assumes
# this notebook sits in code/src/ and the raw CSV sits in code/dataset/.
DATA_PATH = "../dataset/heart_disease.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,Low HDL Cholesterol,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,Yes,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,Yes,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,Yes,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,No,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,No,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No


## 1. Quick Look at the Raw Data

In [3]:
print("Shape (rows, columns):", df.shape)
df.info()

Shape (rows, columns): (10000, 21)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   9971 non-null   float64
 1   Gender                9981 non-null   str    
 2   Blood Pressure        9981 non-null   float64
 3   Cholesterol Level     9970 non-null   float64
 4   Exercise Habits       9975 non-null   str    
 5   Smoking               9975 non-null   str    
 6   Family Heart Disease  9979 non-null   str    
 7   Diabetes              9970 non-null   str    
 8   BMI                   9978 non-null   float64
 9   High Blood Pressure   9974 non-null   str    
 10  Low HDL Cholesterol   9975 non-null   str    
 11  High LDL Cholesterol  9974 non-null   str    
 12  Alcohol Consumption   7414 non-null   str    
 13  Stress Level          9978 non-null   str    
 14  Sleep Hours           9975 non-null   float64
 

## 2. Data Cleaning

### 2.1 Duplicate Rows

In [4]:
n_dup = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dup}")

Fully duplicated rows: 0


No duplicated rows were found, so no rows need to be removed for this reason.

### 2.2 Missing Values

In [5]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_table[missing_table['missing_count'] > 0]

,missing_count,missing_pct
Alcohol Consumption,2586,25.86
Diabetes,30,0.30
Sugar Consumption,30,0.30
Cholesterol Level,30,0.30
Age,29,0.29
Triglyceride Level,26,0.26
CRP Level,26,0.26
High LDL Cholesterol,26,0.26
High Blood Pressure,26,0.26
Low HDL Cholesterol,25,0.25


**Key observation:** `Alcohol Consumption` is missing for **~25.9%** of rows — far more than every other column, which are each missing **under 0.3%** of rows. This column needs a different treatment than the rest:

- **All other columns** (missing < 0.3% each): simply **drop the affected rows** — this only costs a small fraction of the data.
- **`Alcohol Consumption`** (missing ~25.9%): dropping these rows would discard roughly 1 in 4 records, which is too costly. Instead the missing value becomes **its own category (`"Unknown"`)**, after checking it is not strongly associated with the target (so it does not bias the label).

In [6]:
# Does a missing Alcohol Consumption value correlate with the target?
tmp = df.copy()
tmp['Alcohol_missing'] = tmp['Alcohol Consumption'].isnull()
print(tmp.groupby('Alcohol_missing')['Heart Disease Status'].value_counts(normalize=True).round(3))

Alcohol_missing  Heart Disease Status
False            No                      0.797
                 Yes                     0.203
True             No                      0.808
                 Yes                     0.192
Name: proportion, dtype: float64


The disease rate is ~19–20% in both groups (missing vs. not missing), i.e. almost identical to the overall base rate. This supports treating the missing values as effectively random with respect to the label, so recoding them as `"Unknown"` rather than dropping them is a reasonable, low-risk choice.

In [7]:
# Treat missing Alcohol Consumption as its own category
df['Alcohol Consumption'] = df['Alcohol Consumption'].fillna('Unknown')

# For every other column, check how many rows would be affected if dropped
other_cols = [c for c in df.columns if c != 'Alcohol Consumption']
rows_before = len(df)
rows_with_missing_elsewhere = df[other_cols].isnull().any(axis=1).sum()
print(f"Rows with at least one missing value outside Alcohol Consumption: "
      f"{rows_with_missing_elsewhere} ({rows_with_missing_elsewhere/rows_before*100:.2f}% of data)")

Rows with at least one missing value outside Alcohol Consumption: 468 (4.68% of data)


In [8]:
df_clean = df.dropna(subset=other_cols).reset_index(drop=True)
rows_after = len(df_clean)

print(f"Rows before: {rows_before}")
print(f"Rows after:  {rows_after}")
print(f"Rows dropped: {rows_before - rows_after} ({(rows_before-rows_after)/rows_before*100:.2f}%)")
print()
print("Target class ratio BEFORE cleaning:")
print(df['Heart Disease Status'].value_counts(normalize=True).round(4))
print()
print("Target class ratio AFTER cleaning:")
print(df_clean['Heart Disease Status'].value_counts(normalize=True).round(4))

Rows before: 10000
Rows after:  9532
Rows dropped: 468 (4.68%)

Target class ratio BEFORE cleaning:
Heart Disease Status
No     0.8
Yes    0.2
Name: proportion, dtype: float64

Target class ratio AFTER cleaning:
Heart Disease Status
No     0.7995
Yes    0.2005
Name: proportion, dtype: float64


Dropping rows with missing values (outside `Alcohol Consumption`) costs under 5% of the data and leaves the class ratio essentially unchanged (≈80/20 before and after). This confirms the drop is safe.

In [9]:
assert df_clean.isnull().sum().sum() == 0, "There should be no missing values left"
print("Remaining missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Final cleaned shape:", df_clean.shape)

Remaining missing values: 0


Duplicate rows: 0
Final cleaned shape: (9532, 21)


## 3. Export Clean Data

Column names and attribute types (numeric vs. nominal) are left exactly as they are in the
raw CSV — any renaming / ARFF-specific formatting is handled on the Java side
(`DataAnalysis.java`), to keep the Python/Java responsibilities cleanly separated.

In [10]:
OUT_DIR = "../dataset"
os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(OUT_DIR, "cleaned_heart_disease.csv")
df_clean.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df_clean.shape)

Saved: ../dataset\cleaned_heart_disease.csv
Shape: (9532, 21)


## 4. Summary — Step 1 (Python part)

| Step | Decision | Result |
|---|---|---|
| Duplicates | Checked | 0 found |
| Missing values (`Alcohol Consumption`) | Recoded as `"Unknown"` category | 0 rows lost |
| Missing values (all other columns) | Dropped affected rows | 468 rows dropped (4.68%); class ratio unchanged (≈80/20) |

**Final cleaned dataset:** 9,532 rows × 21 columns, 0 missing values, 0 duplicates, saved to `code/dataset/cleaned_heart_disease.csv`.

**Hand-off to Java:** `DataAnalysis.java` reads this file next to continue with Data Analysis (outlier/multicollinearity checks, correlation, chi-square tests), the stratified train/test split, and the ARFF export (`train.arff`, `test.arff`) for Person 2 and Person 3.